In this notebook, we'll build a bigram language model from scratch, and use it to generate some random text. 

# Data: Alice in Wonderland

For this notebook, we will use Lewis Carroll's "Alice in Wonderland", available on Project Gutenberg here: http://www.gutenberg.org/ebooks/11
The plain text version of this book comes with this demo as ```alice.txt```

## Preprocessing the data

We split up the text into sentences, then the sentences into words. As in the "Green eggs and ham" corpus example, we put sentence beginning and end markers around each sentence.

In [3]:
# We read in Alice in Wonderland as a single long string, which we 
# tokenize into sentences. 
# Warning: This may take a while on some computers
import nltk
nltk.download('punkt_tab')

f = open("emma.txt")
alice_text = f.read()
f.close()

import nltk

alice_sents = nltk.sent_tokenize(alice_text)
alice_sents_tokenized = [ nltk.word_tokenize(s) for s in alice_sents]

alice_corpus = []
for st in alice_sents_tokenized:
    alice_corpus = alice_corpus + ["<s>"] + st + ["</s>"] 

alice_bigrams = list(nltk.bigrams(alice_corpus))

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/zichenw/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


We inspect the corpus:

In [4]:
for word in alice_corpus[400:450]: 
    print(word, end =" ")

. </s> <s> CHAPTER XII . </s> <s> CHAPTER XIII . </s> <s> CHAPTER XIV . </s> <s> CHAPTER XV . </s> <s> CHAPTER XVI . </s> <s> CHAPTER XVII . </s> <s> CHAPTER XVIII . </s> <s> CHAPTER XIX . </s> <s> VOLUME I CHAPTER I Emma Woodhouse , 

And the bigrams:

In [5]:
alice_bigrams[600:610]

[('than', 'a'),
 ('a', 'friend'),
 ('friend', ','),
 (',', 'very'),
 ('very', 'fond'),
 ('fond', 'of'),
 ('of', 'both'),
 ('both', 'daughters'),
 ('daughters', ','),
 (',', 'but')]

# Counting words, and computing relative frequencies

## Conditional Frequency Distribution objects in NLTK

The Natural Language Toolkit has data types and functions that make life easier for us when we want to count word n-grams and compute their probabilities using Maximum Likelihood Estimation.

The FreqDist object is a dictionary made for word counting (basically the same as a Counter). Here is an example: 

In [6]:
fd = nltk.FreqDist(alice_bigrams)
fd.most_common(20)

[(('</s>', '<s>'), 6024),
 (('.', '</s>'), 5289),
 ((',', 'and'), 1886),
 (('”', '“'), 987),
 (('’', 's'), 925),
 ((';', 'and'), 867),
 (('<s>', 'I'), 644),
 (('to', 'be'), 594),
 (('of', 'the'), 587),
 ((',', 'I'), 569),
 (('!', '</s>'), 551),
 (('in', 'the'), 452),
 ((',', '”'), 437),
 (('<s>', 'She'), 428),
 ((';', 'but'), 427),
 (('<s>', '“'), 421),
 ((',', '“'), 399),
 (('I', 'am'), 393),
 ((',', 'that'), 361),
 (('!', '—'), 342)]

The ConditionalFreqDist object counts pairs of items, for example: Given that the first word in a pair is 'said', how often is the second word 'Alice'? How often is it 'the'?

In [7]:
# This makes the object
cfd = nltk.ConditionalFreqDist(alice_bigrams)
# This looks up the counts of bigrams starting in "said"
cfd["said"]

FreqDist({'he': 69, ',': 67, 'Emma': 59, 'she': 58, 'Mr.': 38, 'Mrs.': 17, 'I': 15, 'Harriet': 10, 'Frank': 10, 'that': 9, ...})

As you can see, the book has 206 occurrences of "said the", 115 of "said Alice", and so on. 

## Computing bigram frequencies and probabilities 
There is one more data structure we need, NLTK's ConditionalProbDist. This, finally, turns the conditional counts into probabilities. Here is how to build it for Alice in Wonderland: We give it the conditional frequencies, and say that we want MLE probabilities.


In [8]:
# this makes the object
cp = nltk.ConditionalProbDist(cfd, nltk.MLEProbDist)
# this is p(Alice|said)
cp["said"].prob("Alice")

0.0

In [9]:
# Here are all the words that could follow "said"
print("Words that can come after 'said':\n", list(cp["said"].samples()))
# For each word in samples(), we can get its probability
# using prob()
print("And here are their probabilities:")
for word in cp["said"].samples():
    print(word, cp["said"].prob(word))

Words that can come after 'said':
 ['at', 'Emma', 'Mr.', 'her', 'that', 'how', 'it', ',', 'Harriet', 'no', 'of', 'she', 'to', 'he', 'Mrs.', 'the', '—', 'nothing', 'in', 'something', '.', 'before', 'what', 'you', 'John', 'and', 'only', 'a', 'or', 'I', 'Miss', 'Frank', 'for', 'about', 'not', 'so', 'every', 'Jane', ';', 'she.', 'but', '“', 'as', 'just', 'his', 'all', 'much', 'Harriet.', 'such', 'any', 'on', 'enough']
And here are their probabilities:
at 0.0020833333333333333
Emma 0.12291666666666666
Mr. 0.07916666666666666
her 0.0125
that 0.01875
how 0.008333333333333333
it 0.014583333333333334
, 0.13958333333333334
Harriet 0.020833333333333332
no 0.010416666666666666
of 0.004166666666666667
she 0.12083333333333333
to 0.014583333333333334
he 0.14375
Mrs. 0.035416666666666666
the 0.014583333333333334
— 0.010416666666666666
nothing 0.008333333333333333
in 0.014583333333333334
something 0.008333333333333333
. 0.0125
before 0.0020833333333333333
what 0.0020833333333333333
you 0.00625
John 0.0

# Generating text: Probabilistically sampling the next word at each step

Say you roll one fair die. Then each side has a probability of 1/6 of coming up. We can describe this with a *categorical distribution* -- or with a *multinomial distribution*, which describes what happens when you roll k fair dice at once, where we just set k to be 1. The Python *scipy* package has an object that describes a multinomial distribution, called ```stats.multinomial```. It has a function called ```rvs()``` that we can use to roll a die once: It will give us a probabilistic answer, a different one each time we run it. 

Run the following box a few times: 

In [11]:
# Let's roll one die
from scipy import stats

# we have 6 possible outcomes, each with a probability of 1/6
outcomes = [1,2,3,4,5,6]
outcome_probs = [1/6, 1/6, 1/6, 1/6, 1/6, 1/6]
# we only roll one die
num_dice = 1

result = stats.multinomial.rvs(num_dice, outcome_probs)
print("I just rolled one die. Here is what I got.")
# the result we get is a list of 0's and 1's. We get a 1 for
# the number that we rolled
print("Here is what the result looks like out of the box:", result)
# the index of the outcome we got is the 
# place in the result that has a 1
index_of_outcome = list(result).index(1)
print("And here is how we can interpret this:")
print("I rolled a", outcomes[index_of_outcome])


I just rolled one die. Here is what I got.
Here is what the result looks like out of the box: [0 0 0 0 0 1]
And here is how we can interpret this:
I rolled a 6


We can use the same method to probabilistically draw a word. Say we want to draw a bigram that starts in "said...". And say we use the probabilities that we got from the Alice in Wonderland corpus above. Then we can do this:

In [12]:
# possible outcomes: all the words that can follow after "said"
outcomes = list(cp["said"].samples())
# their probabilities, according to the ConditionalProbDist object
outcome_probs = [cp["said"].prob(word) for word in outcomes]
# we only generate a single next word
num_dice = 1

result = stats.multinomial.rvs(num_dice, outcome_probs)
print("I just drew a bigram starting in 'said...'. Here is what I got.")
index_of_outcome = list(result).index(1)
print("said", outcomes[index_of_outcome])

I just drew a bigram starting in 'said...'. Here is what I got.
said in


Now we are ready to generate a longer stretch of text. We start with the sentence start symbol

In [13]:
currentword = "<s>"
print(currentword, end = " ")
# The language model knows, for each word w, 
# the conditional probability P(w|currentword).
# We use that to generate text:
# we start with the sentence-start symbol
for i in range(50):
    # get all the possible words that can follow the current word
    outcomes = list(cp[currentword].samples())
    # and get their probabilities
    outcome_probs = [cp[currentword].prob(word) for word in outcomes]
    # we only sample one next word
    num_dice = 1
    # sample the next word
    result = stats.multinomial.rvs(num_dice, outcome_probs)
    # which outcome index did we sample?
    index_of_outcome = list(result).index(1)
    # The new current word is the word at that index
    currentword = outcomes[ index_of_outcome]
    # print it, and repeat
    print(currentword,end = " ")

<s> Harriet Smith is sure she could not you ; and master appeared. — Have you out . </s> <s> She was heightened by Emma . </s> <s> “ Very like it must extend your usual strength and be prompted it. ” “ No , Miss Bates occupied in general appearances 